# Renters (specialty ltv) v1.5.0 - loss ratio refit validation (cat excluded)Renters schema differs from classic: `projected_ntr_N_lr` carries the applied loss ratio directly, so the propagation check is a direct comparison.**Units:** renters (line 71) is an annual-term product, so everything below is in **NTR (years)**. Nick's original helper was written in `nt6` (a 6-month term counter, because line 16 is a 6-month product) and divided by 2 internally to get years; every renters call site then passed `2 * n` to undo that. Both conversions cancel and have been removed.

In [ ]:
%env ENV_FOR_DYNACONF = prod
%env DYNACONF_GIT_BRANCH = feature/B-2895893
%env DYNACONF_GIT_CHECKOUT = feature/B-2895893

In [ ]:
import ltv_helpers.non_spark_helpers as ns
import pandas as pd
from specialty_ltv import paths as p

pd.options.display.max_rows = 2000
pd.options.display.max_columns = 500

In [ ]:
P_SCORED_NEW = p.score_internal_results
P_SCORED_PRIOR = (
    "tmx-smsiweb/specialty-ltv/prod/NB_SPL_v1.4.0/ltv_calc/score_internal_results/"
)

print(P_SCORED_NEW)

## I. Fits

In [ ]:
LINE = 71
LINE_NAME = "Renters"

# [slope_per_ntr, intercept]
SPL_FIT = [-0.043571, 0.625204]
PROD_FIT = [-0.05045, 0.7259]

# was MAX_NT6 = 18. Inert over NTR 0-9, but the NTR=9 censored bucket is still
# an open item -- do not read "cap never binds here" as "censoring is handled".
MAX_NTR = 9
N_NTR = 10


def eval_fit(ntr, fit):
    slope, intercept = fit
    return intercept + slope * min(ntr, MAX_NTR)


def lr_summary(path):
    d = ns.read_parquet_s3_to_pandas(path)[
        ["drv_line", "lifetime_premium", "lifetime_loss", "cat_loss_amt", "balance_amt"]
    ]
    g = d.groupby("drv_line", as_index=False).agg(
        premium=("lifetime_premium", "sum"),
        loss=("lifetime_loss", "sum"),
        cat=("cat_loss_amt", "sum"),
        bal=("balance_amt", "sum"),
    )
    g["lr_ex_cat"] = g["loss"] / g["premium"]
    g["lr_total"] = (g["loss"] + g["cat"]) / g["premium"]
    g["cat_share"] = g["cat"] / (g["loss"] + g["cat"])
    return g

### Fit deltaPreviously this ran `n in range(10)` through the nt6 path, so it only covered NTR 0-4.5 while every other cell treated `n` as NTR. Now it covers the full scored range.

In [ ]:
pd.DataFrame(
    [
        {
            "ntr": n,
            "prod": eval_fit(n, PROD_FIT),
            "new": eval_fit(n, SPL_FIT),
        }
        for n in range(N_NTR)
    ]
).assign(diff=lambda d: d["new"] - d["prod"]).round(4)

## II. Load

In [ ]:
LR_COLS = [f"projected_ntr_{n}_lr" for n in range(N_NTR)]

COLS = [
    "drv_line",
    "lifetime_premium",
    "lifetime_loss",
    "cat_loss_amt",
    "balance_amt",
    "bal_factor",
    "lr_cat",
    "loss_ratio_x_cat_yr1_target",
    "loss_ratio_x_cat_yr2_target",
    "loss_ratio_x_cat_yr3_target",
] + LR_COLS
# dropped: ply_pt_state_cd, ply_ntr_nbr, premium_new, premium_renew -- loaded but never
# read. Any state-level carve-out would trip the nunique assertion in III anyway.

df = ns.read_parquet_s3_to_pandas(P_SCORED_NEW)[COLS]
df.shape

## III. Propagation checkThe three previous cells (obs vs nt6/ntr, the `IDX` gap table, the vs-prod table) were the same10-number comparison run three times over ~2M rows. Collapsed into one. `obs_nunique` was 1 forevery column, so the applied loss ratio is a pure function of NTR -- that is now asserted ratherthan eyeballed, and one row is enough to compare against.

In [ ]:
assert df[LR_COLS].isna().sum().sum() == 0, "null projected loss ratios"

nuniq = df[LR_COLS].nunique()
assert (nuniq == 1).all(), nuniq[nuniq != 1]  # fails loudly if a carve-out ever appears

chk = pd.DataFrame(
    {
        "ntr": range(N_NTR),
        "obs": df[LR_COLS].iloc[0].to_numpy(),
        "expected_new": [eval_fit(n, SPL_FIT) for n in range(N_NTR)],
        "expected_prod": [eval_fit(n, PROD_FIT) for n in range(N_NTR)],
    }
)
chk["gap"] = (chk["obs"] - chk["expected_new"]).abs()
chk["vs_prod"] = chk["obs"] - chk["expected_prod"]
chk["pass"] = chk["gap"] < 1e-4

assert chk["pass"].all(), chk[~chk["pass"]]
chk.round(6)

## IV. Cat components`lr_cat` is applied as a single constant against premium, so dividing `cat_loss_amt` back bypremium can only return `lr_cat`. Test the construction directly instead of reporting thequotient as if it were an independent measurement.

In [ ]:
m = df["lifetime_premium"] != 0
resid = (df.loc[m, "cat_loss_amt"] / df.loc[m, "lifetime_premium"] - df.loc[m, "lr_cat"]).abs().max()
print(f"max |cat_loss_amt/premium - lr_cat| = {resid:.3e}  (rows with premium=0: {(~m).sum()})")

TGT = [f"loss_ratio_x_cat_yr{i}_target" for i in (1, 2, 3)]
print("target nunique:", df[TGT].nunique().to_dict())
print("max row-wise spread yr1/yr2/yr3:", df[TGT].std(axis=1).max())

df[["lr_cat"] + TGT + ["bal_factor"]].mean().round(4)

## V. v1.5.0 vs v1.4.0`bal` is reported for reference only -- `balance_amt` does not enter `lr_ex_cat` or `lr_total`.`d_bal` is large (~6e7), so either it belongs in the loss numerator or it should come out ofthis summary. Open question, not resolved here.

In [ ]:
cmp = lr_summary(P_SCORED_NEW).merge(
    lr_summary(P_SCORED_PRIOR), on="drv_line", suffixes=("_new", "_old")
)

for c in ["lr_ex_cat", "lr_total", "premium", "loss", "cat", "bal"]:
    cmp[f"d_{c}"] = cmp[f"{c}_new"] - cmp[f"{c}_old"]
cmp["prem_pct"] = cmp["d_premium"] / cmp["premium_old"]

cmp[
    [
        "drv_line",
        "lr_ex_cat_old",
        "lr_ex_cat_new",
        "d_lr_ex_cat",
        "lr_total_old",
        "lr_total_new",
        "d_lr_total",
        "d_cat",
        "d_bal",
        "d_premium",
        "prem_pct",
    ]
].round(4)

### Did the cat component change?`cat_comp = cat / premium` on both sides, and `cat = premium * lr_cat` with `lr_cat` unchanged,so `d_cat_comp = 0` falls out by construction -- it is not evidence of anything. The informativequantity is `implied`: what `lr_ex_cat` would be if the *only* change were stripping the catshare out of the old total. `refit_effect` is the part the refit moved beyond that.

In [ ]:
cmp["implied"] = cmp["lr_ex_cat_old"] * (1 - cmp["cat_share_new"])
cmp["refit_effect"] = cmp["lr_ex_cat_new"] - cmp["implied"]

cmp[
    [
        "drv_line",
        "lr_ex_cat_old",
        "cat_share_new",
        "implied",
        "lr_ex_cat_new",
        "refit_effect",
    ]
].round(4)